In [1]:
import furuta_systems
import pandas as pd
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
import furuta_params
import furuta_trajectory_gen
from IPython.display import display, Math

# Überführung in der stabilen Ruhelage

#### Zunächst wird die Überführung des Rotorarms in der stabilen Ruhelage des Pendelarms von einer in eine andere Position betrachtet. Um diese Ruhelagenüberführung durchführen zu können, ist dass in Aufgabe 3.1 erarbeitete nichtlineare Modell zu verwenden. Gehen Sie zunächst davon aus, dass keine Pendelmasse mP am Pendelarm angebracht ist und vernachlässigen Sie die im System auftretenden Haftreibungen, mu_H,1 = mu_H,2 = 0.

## 3.8 Geben Sie zunächst allgemein die Ruhelagen des Systems an.

In [2]:
display(Math(r"\tau = 0"))

display(Math(r"m_2 \, l_2 \, g \, \sin(\theta_2) = 0"))

display(Math(r"\theta_2 = 0 \;\text{oder}\; \theta_2 = \pi"))

display(Math(r"\theta_1 \;\text{ist beliebig}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3.9 Für den Steuerungsentwurf linearisieren Sie das nichtlineare Modell des Furuta Pendels um die stabile Ruhelage

### a) Berechnen Sie die Systemmatrix A und den Eingangsvektor b mit den in Tab. 1 gegebenen Parametern.

In [3]:
params_auslegung = furuta_params.FurutaParams()
params_auslegung.mu_H1 = 0.0
params_auslegung.mu_H2 = 0.0
tau_max = params_auslegung.tau_max
operating_point = (0,0,0,0,0)

In [4]:
A, b = furuta_systems.get_linear_furuta(params_auslegung, operating_point)
print("A = \n" + 
    np.array2string(
    A,
    precision=3,
    suppress_small=True,
    max_line_width=120
))
print("b = \n" + 
    np.array2string(
    b,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

A = 
[[  0.      1.      0.      0.   ]
 [  0.    -17.516   1.075   0.   ]
 [  0.      0.      0.      1.   ]
 [  0.     16.969 -62.354  -0.026]]
b = 
[[  0.   ]
 [ 34.406]
 [  0.   ]
 [-33.331]]


### b) Überprüfen Sie die Steuerbarkeit des linearisierten Modells.

In [10]:
n = A.shape[0]

# Steuerbarkeitsmatrix S = [B, AB, A^2B, ...]
S_blocks = [np.linalg.matrix_power(A, k) @ b for k in range(n)] #liste von spalten vektoren
S_np = np.hstack(S_blocks) #spaletenvektoren werden zu matrix zusammen gefügt

rankS = np.linalg.matrix_rank(S_np)  # nutzt intern toleranz

print("rank(S):", rankS)

rank(S): 4


S hat vollen Rang und das System ist entsprechend Steuerbar.

### c) Berechnen Sie die Eigenwerte der Systemmatrix.

In [16]:
eigvals = np.linalg.eigvals(A)
print("Liste von Eigenwerten: \n" + 
    np.array2string(
    eigvals,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

Liste von Eigenwerten: 
[  0.   +0.j    -17.467+0.j     -0.038+7.841j  -0.038-7.841j]


### d) Überführen Sie das linearisierte System in Regelungsnormalform (RNF) und führen Sie einen flachen Ausgang ein.

In [19]:
A_RNF, Q = furuta_systems.get_RNF_furuta(params_auslegung, operating_point)
b_RNF = Q @ b

print("A_RNF = \n" + 
    np.array2string(
    A_RNF,
    precision=3,
    suppress_small=True,
    max_line_width=120
))
print("b_RNF = \n" + 
    np.array2string(
    b_RNF,
    precision=3,
    suppress_small=True,
    max_line_width=120
))

A_RNF = 
[[    0.        1.        0.        0.   ]
 [    0.        0.        1.        0.   ]
 [    0.        0.        0.        1.   ]
 [    0.    -1073.96    -62.798   -17.542]]
b_RNF = 
[[0.]
 [0.]
 [0.]
 [1.]]


### e) Erarbeiten Sie auf Basis der RNF eine Steuerung, die zur Überführung zwischen stationären Regimen innerhalb eines vorgegebenes Zeitintervall geeignet ist.


In [29]:
# Symbole
n, k = sp.symbols('n k', integer=True, nonnegative=True)
tau = sp.symbols('tau', nonzero=True)

# Formel als SymPy-Ausdruck (allgemein)
phi = sp.factorial(2*n + 1) / sp.factorial(n)**2 * sp.summation(
    sp.binomial(n, k) * (-1)**k * tau**(n + k + 1) / (n + k + 1),
    (k, 0, n)
)

# Beispiel: n = 4
phi4 = sp.simplify(phi.subs(n, 4))
phi4 
        

Piecewise((126*tau**5*hyper((-4, 5), (6,), tau), (tau >= -1) & (tau <= 1)), (tau**5*(70*tau**4 - 315*tau**3 + 540*tau**2 - 420*tau + 126), True))


Piecewise((126*tau**5*hyper((-4, 5), (6,), tau), (tau >= -1) & (tau <= 1)), (70*tau**9 - 315*tau**8 + 540*tau**7 - 420*tau**6 + 126*tau**5, True))

Prototyp Polynom Phi (die Beiden Ausdrücke sollten equvalent sein) - für alles weitere siehe Aufgabe 3.10